In [1]:
print("hi")

hi


In [2]:
# !uv pip install graphiti-core[falkordb]==v0.17.7
# !uv pip install pandas

In [3]:
import os
import json
import requests
import pandas as pd
from io import StringIO
from bs4 import BeautifulSoup
from graphiti_core import Graphiti
from datetime import datetime, timezone
from graphiti_core.nodes import EpisodeType
from graphiti_core.driver.falkordb_driver import FalkorDriver
from graphiti_core.search.search_config_recipes import NODE_HYBRID_SEARCH_RRF

# Configuration
os.environ["MODEL_NAME"] = "gpt-4.1"
group_id = "la-liga"

In [4]:
async def add_episodes_to_graph(graphiti, episodes, group_id, prefix="Episode"):
    print(f"📝 Adding {len(episodes)} episodes to graph...")
    for i, episode in enumerate(episodes):
        name = episode.get('name',f"{prefix} {i+1}")
        content = episode['content']

        if not isinstance(content,str):
            content = json.dumps(content)

        await graphiti.add_episode(
            name=name,
            episode_body=content,
            source=episode['type'],
            source_description=episode['description'],
            reference_time=datetime.now(timezone.utc),
            group_id=group_id
        )
    print(f"✅ Successfully added {len(episodes)} episodes!")

def get_standing_table(url, teams_filter, date):
    """Extract La Liga standings from Wikipedia."""
    print(f"🏆 Fetching standings data for {date}...")
    headers =  {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"}
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    standing_table = pd.read_html(StringIO(response.text))[5]

    standing_table = standing_table.rename(columns={
        "Teamvte": "TeamName",
        "Pts": "Season Points",
        "Pos": "Season Position",
        "W": "Season Wins"
    })
    standing_table = standing_table[["TeamName", "Season Points", "Season Position", "Season Wins"]]
    if not standing_table.empty and " (C)" in standing_table.at[0, "TeamName"]:
        standing_table.at[0, "TeamName"] = standing_table.at[0, "TeamName"].replace(" (C)", "")
    
    standing_table['Relevant Period']=date

    episodes = []
    for row in standing_table.to_dict(orient="records"):
        if row["TeamName"] in teams_filter:
            episode = {
                "content": row,
                "type": EpisodeType.json,
                "description": f"Extract the {date} La Liga Standing into entities",
            }
            episodes.append(episode)
    
    print(f"✅ Found {len(episodes)} relevant team standings")
    return episodes

def get_topscorers_table(url, teams_filter, date):
    """Extract top scorers data from Wikipedia."""
    print(f"⚽ Fetching top scorers data for {date}...")

    headers =  {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"}
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    top_soccer_table = pd.read_html(StringIO(response.text))[7]
    top_soccer_table = top_soccer_table.rename(columns={
        "Goals[51]": "Season Goals"
    })
    episodes = []
    for row in top_soccer_table.to_dict(orient="records"):
        if row["Club"] in teams_filter:
            episode = {
                "content": row,
                "type": EpisodeType.json,
                "description": f"Extract the {date} La Liga top player stats into different entities",
            }
            episodes.append(episode)
    print(f"✅ Found {len(episodes)} relevant top scorers")
    return episodes

def get_article_from_url(url):
    """Scrape article content from a URL."""
    print(f"📰 Fetching article from: {url}")
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')

    date_meta = soup.find("meta",{"name": "DC.date.issued"})
    article_date = date_meta["content"] if date_meta and date_meta.get("content") else "Date not found"

    paragraphs = soup.find_all("p")
    filtered = [p.get_text(strip=True) for p in paragraphs if len(p.get_text(strip=True))>50]

    article_text = "\n\n".join(filtered).encode("utf-8", "ignore").decode("utf-8")
    article_text = article_text.replace("í", "i")

    print("✅ Article extracted successfully")
    return article_date, article_text

async def search_and_display(graphiti, query, num_results=5):
    """Search the graph and display results in a clean format."""
    print(f"🔍 Searching for: '{query}'")
    print("-" * 50)

    results = await graphiti.search(query, num_results=num_results)
    for i, r in enumerate(results):
        print(f"{i}. {r.fact}")
        print(f"   Label: {r.name}")
        print(f"   📅 Valid from: {r.valid_at}")
        if r.invalid_at:
            print(f"   ❌ Invalid at: {r.invalid_at}")
        print()

    return results        


In [5]:
from graphiti_core.llm_client.openai_client import OpenAIClient
from openai import AsyncAzureOpenAI
from graphiti_core.embedder.openai import OpenAIEmbedder
from graphiti_core.cross_encoder.openai_reranker_client import OpenAIRerankerClient

client = AsyncAzureOpenAI(
    api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT")
)
embedder = OpenAIEmbedder(client=client)
llm_client = OpenAIClient(client=client)
cross_encoder = OpenAIRerankerClient(client=client)
falkor_driver = FalkorDriver(
    host="r-6jissuruar.instance-1pq0d4red.hc-7up0crkyn.ap-south-1.aws.f2e0a955bb84.cloud",
    port="63666",
    username="falkordb",
    password="12345678",
    database=group_id,
)
graphiti = Graphiti(
    graph_driver=falkor_driver,
    llm_client=llm_client,
    embedder=embedder,
    cross_encoder=cross_encoder
)
await graphiti.build_indices_and_constraints()

#### Part 1: Structured Data - La Liga Standings

In [6]:
# !uv pip install lxml

In [7]:
standing_2324_url = 'https://en.wikipedia.org/wiki/2023%E2%80%9324_La_Liga'
TARGET_TEAMS = {"Real Madrid", "Barcelona"}
relevant_date = '2023-07-25-2024-05-25'

episodes_2324 = get_standing_table(standing_2324_url, TARGET_TEAMS, relevant_date)

print("\n📋 2023/24 Season Data:")
for episode in episodes_2324:
    content = episode['content']
    print(f"Team: {content['TeamName']}, Points: {content['Season Points']}, Position: {content['Season Position']}")

await add_episodes_to_graph(graphiti, episodes_2324, group_id, prefix="LaLiga 23/24 Standings")


🏆 Fetching standings data for 2023-07-25-2024-05-25...
✅ Found 2 relevant team standings

📋 2023/24 Season Data:
Team: Real Madrid, Points: 95, Position: 1
Team: Barcelona, Points: 85, Position: 2
📝 Adding 2 episodes to graph...
✅ Successfully added 2 episodes!


In [8]:
episodes_2324

[{'content': {'TeamName': 'Real Madrid',
   'Season Points': '95',
   'Season Position': 1,
   'Season Wins': 29,
   'Relevant Period': '2023-07-25-2024-05-25'},
  'type': <EpisodeType.json: 'json'>,
  'description': 'Extract the 2023-07-25-2024-05-25 La Liga Standing into entities'},
 {'content': {'TeamName': 'Barcelona',
   'Season Points': '85',
   'Season Position': 2,
   'Season Wins': 26,
   'Relevant Period': '2023-07-25-2024-05-25'},
  'type': <EpisodeType.json: 'json'>,
  'description': 'Extract the 2023-07-25-2024-05-25 La Liga Standing into entities'}]

In [9]:
standing_2425_url = 'https://en.wikipedia.org/wiki/2024%E2%80%9325_La_Liga'
relevant_date = '2024-07-25-2025-05-25'

episodes_2425 = get_standing_table(standing_2425_url, TARGET_TEAMS, relevant_date)

print("\n📋 2024/25 Season Data:")
for episode in episodes_2425:
    content = episode['content']
    print(f"Team: {content['TeamName']}, Points: {content['Season Points']}, Position: {content['Season Position']}")

await add_episodes_to_graph(graphiti, episodes_2425, group_id, prefix="LaLiga 24/25 Standings")

🏆 Fetching standings data for 2024-07-25-2025-05-25...
✅ Found 2 relevant team standings

📋 2024/25 Season Data:
Team: Barcelona, Points: 88, Position: 1
Team: Real Madrid, Points: 84, Position: 2
📝 Adding 2 episodes to graph...
✅ Successfully added 2 episodes!


In [10]:
top_scorers_url = 'https://en.wikipedia.org/wiki/2024%E2%80%9325_La_Liga'
relevant_date = 'Season 24/25'

episodes_scorers = get_topscorers_table(top_scorers_url, TARGET_TEAMS, relevant_date)
await add_episodes_to_graph(graphiti, episodes_scorers, group_id, prefix="LaLiga 24/25 Top Scorers")


⚽ Fetching top scorers data for Season 24/25...
✅ Found 3 relevant top scorers
📝 Adding 3 episodes to graph...
✅ Successfully added 3 episodes!


In [11]:
fact_results = await search_and_display(graphiti, "What was Real Madrid’s final points at the end of each season?")

🔍 Searching for: 'What was Real Madrid’s final points at the end of each season?'
--------------------------------------------------
0. Real Madrid had 95 season points in the period 2023-07-25 to 2024-05-25.
   Label: HAS_SEASON_POINTS
   📅 Valid from: 2023-07-25 00:00:00+00:00
   ❌ Invalid at: 2024-05-25 00:00:00+00:00

1. Real Madrid has 84 season points in the 2024-07-25 to 2025-05-25 period.
   Label: HAS_SEASON_POINTS
   📅 Valid from: 2026-05-24 14:16:58+00:00

2. Real Madrid had season position 1 in the period 2023-07-25 to 2024-05-25.
   Label: HAS_SEASON_POSITION
   📅 Valid from: 2023-07-25 00:00:00+00:00
   ❌ Invalid at: 2024-05-25 00:00:00+00:00

3. Real Madrid had 29 season wins in the period 2023-07-25 to 2024-05-25.
   Label: HAS_SEASON_WINS
   📅 Valid from: 2023-07-25 00:00:00+00:00
   ❌ Invalid at: 2024-05-25 00:00:00+00:00

4. Real Madrid has 26 season wins in the 2024-07-25 to 2025-05-25 period.
   Label: HAS_SEASON_WINS
   📅 Valid from: 2026-05-24 14:16:58+00:00



In [12]:
article_url = "https://www.espn.com/soccer/story/_/id/45783151/marcus-rashford-arrives-barcelona-loan-man-united"
article_date, article_text = get_article_from_url(article_url)

espn_episode = {
    'content': f"{article_date}\n\n{article_text}",
    'type': EpisodeType.text,
    'description': "Football transfer news and rumors"
}

print("\n📰 Article Preview:")
print(article_text[:800] + "...\n")

await add_episodes_to_graph(graphiti, [espn_episode], group_id, prefix="ESPN Transfer News")


📰 Fetching article from: https://www.espn.com/soccer/story/_/id/45783151/marcus-rashford-arrives-barcelona-loan-man-united
✅ Article extracted successfully

📰 Article Preview:
Mark Ogden discusses Marcus Rashford's potential move to Barcelona after the Spanish club were given permission to speak to the player. (2:01)

Marcus Rashfordlanded inBarcelonaon Sunday ahead of completing a season-long loan move fromManchester United.

Rashford, 27, will undergo a medical early in the week and, if everything goes to plan, will be presented as a Barça player before the club head off on tour, a source told ESPN.

Barça fly to Asia on Thursday and coach Hansi Flick was keen to have Rashford with the team in Japan and South Korea to give him as much time as possible to bed in before the season starts in August.

- Sources:Rashford close to Barcelona loan move-Nico Williams explains 10-year Athletic extension- Sources:Man Utd close on Mbeumo before U.S. tour

Rashford was cle...

📝 Adding 1 episodes

In [13]:
fact_results = await search_and_display(graphiti, "Who are the players rumored to move to Barcelona?")

🔍 Searching for: 'Who are the players rumored to move to Barcelona?'
--------------------------------------------------
0. Barcelona had made signing Luis Diaz their priority this summer.
   Label: TRANSFER_PRIORITY_OF
   📅 Valid from: 2025-07-20 00:00:00+00:00

1. Mark Ogden discusses Marcus Rashford's potential move to Barcelona after the Spanish club were given permission to speak to the player.
   Label: DISCUSSES_POTENTIAL_MOVE_TO
   📅 Valid from: 2025-07-20 20:37:00+00:00

2. Marcus Rashford has always preferred a move to Barcelona after the possibility of such a transfer emerged in January.
   Label: PREFERRED_MOVE_TO
   📅 Valid from: 2025-01-01 00:00:00+00:00

3. Robert Lewandowski plays for Barcelona.
   Label: PLAYS_FOR
   📅 Valid from: 2026-05-24 14:17:34+00:00

4. Barcelona had made signing a left winger their priority this summer, including Marcus Rashford.
   Label: TRANSFER_PRIORITY_OF
   📅 Valid from: 2025-07-20 00:00:00+00:00



In [14]:
# Configure node search
node_search_config = NODE_HYBRID_SEARCH_RRF.model_copy(deep=True)
node_search_config.limit = 2

# Search for specific entity
node_search_results = await graphiti._search(
    query='Hansi Flick',
    config=node_search_config
)

print("🔍 Node Search Results:")
print("-" * 30)
for i, node in enumerate(node_search_results.nodes, 1):
    print(f"{i}. Name: {node.name}")
    print(f"   Summary: {node.summary[:200]}...")
    print()

🔍 Node Search Results:
------------------------------
1. Name: Hansi Flick
   Summary: Hansi Flick is a football coach who was keen to have Marcus Rashford join Barcelona's team in Japan and South Korea to help him integrate before the season starts. The messages do not provide addition...

2. Name: Robert Lewandowski
   Summary: Marcus Rashford, a 27-year-old English international footballer, is set to join Barcelona on a season-long loan from Manchester United. The move includes an option for a permanent transfer next summer...



In [15]:
fact_results = await search_and_display(graphiti, "What the latest news about Manchester United")

🔍 Searching for: 'What the latest news about Manchester United'
--------------------------------------------------
0. Marcus Rashford spent the last two weeks training away from the first-team squad at Manchester United after being told by coach Ruben Amorim that he does not feature in his plans.
   Label: TRAINED_AWAY_FROM_FIRST_TEAM_AT
   📅 Valid from: 2025-07-06 00:00:00+00:00
   ❌ Invalid at: 2025-07-20 00:00:00+00:00

1. Marcus Rashford was told by coach Ruben Amorim that he does not feature in his plans at Manchester United.
   Label: COACHED_BY
   📅 Valid from: 2025-07-06 00:00:00+00:00
   ❌ Invalid at: 2025-07-20 00:00:00+00:00

2. Marcus Rashford last played for Manchester United against Viktoria Plzen in the Europa League last December.
   Label: PLAYED_AGAINST
   📅 Valid from: 2024-12-01 00:00:00+00:00
   ❌ Invalid at: 2025-07-20 00:00:00+00:00

3. Mark Ogden discusses Marcus Rashford's potential move to Barcelona after the Spanish club were given permission to speak to the 

In [16]:
# Simulate new transfer updates
new_updates = [
    {
        "content": "Lionel Messi is rumored to be transferring to Barcelona from Inter Miami.",
        "type": EpisodeType.message,
        "description": "Latest transfer rumor update"
    },
    {
        "content": "Mark Ogden reports that Marcus Rashford has renewed his contract with Manchester United until 2028 and he is no longer connected to any move or loan to Barcelona anymore.",
        "type": EpisodeType.message,
        "description": "Previous facts updates - update the old facts"
    }
]

# Add updates to graph
await add_episodes_to_graph(graphiti, new_updates, group_id, prefix="Transfer Update")

📝 Adding 2 episodes to graph...
✅ Successfully added 2 episodes!


In [17]:
fact_results = await search_and_display(graphiti, "Who are the players rumored to move to Barcelona?")

🔍 Searching for: 'Who are the players rumored to move to Barcelona?'
--------------------------------------------------
0. Lionel Messi is rumored to be transferring to Barcelona from Inter Miami.
   Label: RUMORED_TO_TRANSFER_FROM_TO
   📅 Valid from: 2026-05-24 14:22:34+00:00

1. Barcelona had made signing Luis Diaz their priority this summer.
   Label: TRANSFER_PRIORITY_OF
   📅 Valid from: 2025-07-20 00:00:00+00:00

2. Mark Ogden discusses Marcus Rashford's potential move to Barcelona after the Spanish club were given permission to speak to the player.
   Label: DISCUSSES_POTENTIAL_MOVE_TO
   📅 Valid from: 2025-07-20 20:37:00+00:00

3. Marcus Rashford has always preferred a move to Barcelona after the possibility of such a transfer emerged in January.
   Label: PREFERRED_MOVE_TO
   📅 Valid from: 2025-01-01 00:00:00+00:00
   ❌ Invalid at: 2025-07-20 00:00:00+00:00

4. Robert Lewandowski plays for Barcelona.
   Label: PLAYS_FOR
   📅 Valid from: 2026-05-24 14:17:34+00:00



In [18]:
fact_results = await search_and_display(graphiti, "What is the latest news about Manchester United?")

🔍 Searching for: 'What is the latest news about Manchester United?'
--------------------------------------------------
0. Mark Ogden reports that Marcus Rashford has renewed his contract with Manchester United until 2028.
   Label: REPORTED_BY
   📅 Valid from: 2026-05-24 14:22:42+00:00

1. Marcus Rashford has renewed his contract with Manchester United until 2028.
   Label: RENEWED_CONTRACT_WITH
   📅 Valid from: 2026-05-24 14:22:42+00:00
   ❌ Invalid at: 2028-01-01 00:00:00+00:00

2. Marcus Rashford spent the last two weeks training away from the first-team squad at Manchester United after being told by coach Ruben Amorim that he does not feature in his plans.
   Label: TRAINED_AWAY_FROM_FIRST_TEAM_AT
   📅 Valid from: 2025-07-06 00:00:00+00:00
   ❌ Invalid at: 2025-07-20 00:00:00+00:00

3. Marcus Rashford last played for Manchester United against Viktoria Plzen in the Europa League last December.
   Label: PLAYED_AGAINST
   📅 Valid from: 2024-12-01 00:00:00+00:00
   ❌ Invalid at: 2025-